# Testing Sparse Auto Encoder (SAE)

In principal, it should be possible to extract hidden feature vectors from embedding outputs using a sparse auto-encoder (SAE) to disentangle the superimposed features for a given set of neural activation outputs. To test the validity of using a SAE for this purpose, the code below trains one on some randomized vectors to assess whether the methods described in literature are enough to create sparsity in the feature layer of a 2-layer autoencoder.

*Note - This notebook was originally run on a gcloud instance with 32GB of memory. If you intend to re-run this notebook you will need similar compute and memory availability to avoid kernel crashes*

In [2]:
# imports
import tensorflow as tf
import keras
from keras import layers
import numpy as np
import pandas as pd
import Bio
import transformers
from objects.autoencoder import SparseAutoEncoder
import os

# data vis packages
import seaborn as sns
import matplotlib.pyplot as plt

### Train the Model

In [ ]:
# create dataset from random tensors to test
SAE_name = 'autoencoder_test'
embed_length = 1024
ef = 4

print("=== Generating Test Data ===")
fake_embeddings = tf.random.uniform(shape=[10000, embed_length])
fake_dataset = tf.data.Dataset.from_tensor_slices((fake_embeddings, fake_embeddings)).batch(100)

print("=== Initializing Model ===")
# initialize the optimizer
optimizer = keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.98,
                                     epsilon=1e-9)

# initialize the loss function
loss = keras.losses.MeanSquaredError()

# initialize the metrics
metrics = [
    keras.metrics.MeanSquaredError(),
    keras.metrics.Metric(name='placeholder') # placeholder for training, feature output requires a 2nd metric to appease keras
]

# compile the model
autoencoder = SparseAutoEncoder(encoding_size=embed_length, expansion_factor=ef)
autoencoder.compile(optimizer=optimizer, loss=loss, metrics=metrics)

tb_callback = keras.callbacks.TensorBoard(log_dir=f'./logs/{SAE_name}', histogram_freq=5)
early_stopping = keras.callbacks.EarlyStopping(
    monitor='mean_squared_error',
    min_delta=0.001,
    patience=10, 
    restore_best_weights=True
    )

print("=== Training Model ===")
history = autoencoder.fit(fake_dataset, epochs=100, callbacks=[tb_callback, early_stopping])

print("=== Saving Model ===")
path = f'./models/{SAE_name}.keras'

autoencoder.save(path)
autoencoder.save_weights(f'./models/{SAE_name}.weights.h5') # workaround via weight saving / loading

print(f"Model saved to: {path}")
print(autoencoder.summary())

=== Generating Test Data ===
=== Initializing Model ===
=== Training Model ===
Epoch 1/100


I0000 00:00:1731949569.674915   37072 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1731949569.707194   37072 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1731949569.707484   37072 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1731949569.709076   37072 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

 13/100 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 2241.8391 - mean_squared_error: 0.3135

I0000 00:00:1731949572.548384   37640 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


100/100 ━━━━━━━━━━━━━━━━━━━━ 6s 35ms/step - loss: 1439.4963 - mean_squared_error: 0.3220
Epoch 2/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 327.3087 - mean_squared_error: 0.2328
Epoch 3/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 353.6745 - mean_squared_error: 0.1506
Epoch 4/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 341.7060 - mean_squared_error: 0.1129
Epoch 5/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 337.4852 - mean_squared_error: 0.1843
Epoch 6/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - loss: 337.1566 - mean_squared_error: 0.2148
Epoch 7/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 336.6304 - mean_squared_error: 0.1867
Epoch 8/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 339.5449 - mean_squared_error: 0.1751
Epoch 9/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 336.5508 - mean_squared_error: 0.2080
Epoch 10/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 342.6860 - mean_squared_error: 0.1862
Epoch 11/100

...

 Total params: 100,663,298 (384.00 MB)

 Trainable params: 33,554,432 (128.00 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 67,108,866 (256.00 MB)

None


### Examine Weights and their Distribution

Note that in a real use case we would expect most features to have relatively low weights (roughly corresponding to the feature's importance), however, in this case we observe a somewhat sine-like distribution with a bit of decay. I suspect this is due to the process of random number generation, but further analysis would be warranted before asserting that claim.

As a note to myself, the concept of superposition shares a lot of common principles with slit experiments and waves so this particular graph might actually warrant some further investigation from a purely mathematical standpoint. 

In [11]:
feat_weights = np.array(autoencoder.weights[1])
feat_weights = feat_weights.flatten()
print(feat_weights.max())
print(feat_weights.min())

0.08939428
-0.030460445


In [ ]:
sns.histplot(feat_weights, kde=True)
plt.title('Distribution of Feature Weights')
plt.xlabel('Feature Weight')
plt.ylabel('Frequency')
plt.show()

: 

### Examining Feature Layer for Sparsity

In [ ]:
autoencoder.predict_on_batch(fake_embeddings[:100])

In [ ]:
reconstructed_outputs, feature_outputs = autoencoder.predict_on_batch(fake_embeddings[:100])

In [ ]:
reconstructed_outputs

In [ ]:
feature_outputs

In [ ]:
feature_outputs[0].sort()
print(feature_outputs[0][-20:])
print(feature_outputs.shape)

In [ ]:
sns.histplot(feature_outputs.flatten(), kde=True)
plt.title('Distribution of feature_outputs')
plt.xlabel('Feature Value')
plt.ylabel('Frequency')
plt.show()

### Check Model Fit / Overfit

In [ ]:
# check model fit
fit_test_data = tf.random.uniform(shape=[100, embed_length])
fit_test_dataset = tf.data.Dataset.from_tensor_slices((fit_test_data, fit_test_data)).batch(10)

fit_test = autoencoder.evaluate(fit_test_dataset)
print(f"Loss: {fit_test[0]} | MSE: {fit_test[1]}")

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['mean_squared_error'])
plt.show()
plt.plot(history.history['loss'])
plt.show()

In [ ]:
# cleaning some memory
del fake_embeddings
del fake_dataset
del feat_weights
del reconstructed_outputs, feature_outputs, fit_test, fit_test_data, fit_test_dataset
del autoencoder, loss, metrics, optimizer, SAE_name, embed_length, ef, tb_callback, early_stopping, history, path

# SAE on ESM Embeddings for Hemoglobin Variants

Using the embeddings we previously generated from ESM-2, we're going to extract per-residue features based on the superposition hypothesis. After a bit of reformatting the data for this purpose, we should be able to train it to decompose the 1280-dimensional embeddings into a given number of features (number of neurons in the feature/2nd layer). The weights approximate the importance of those features.

In [ ]:
# load metadata
var_data = pd.read_csv('./data/hbvar-w-func-embed-seq.csv')
# pandas saves these as strings for some reason - reload from numpy
var_data.drop(columns=['embeddings', 'sequence_representation', 'esm_representation'], inplace=True)
# drop stuff we don't need right now
var_data.drop(columns=[
    'electrophoresisComment',
    'hgvsName',
    'chromatography',
    'dnaDescriptionComment',
    'aliases',
    'stability',
    'functionalComment',
    'variantComment',
    'Structure',
    'electrophoresis',
    'name'
    ],
    inplace=True
)

# explicitly match embeddings to sequences when re-loading
embeddings = [np.load(f"./data/variant_embeddings/all_vars/{j}.npy") for j in var_data["sequence"]]

# reduce jagged tensors and add to dataframe
def fix_jagged_2d(embed_list):
    sir_gallahad_the_chaste = []
    # find max length in 2nd dim (sequence length)
    max_len = max([len(arr[0]) for arr in embed_list])
    # append zero vector to embeddings
    for i, arr in enumerate(embed_list):
        if len(arr[0]) < max_len:
            padded = np.append(arr[0], np.zeros((max_len - len(arr[0]), 1280)), axis=0)
            sir_gallahad_the_chaste.append(np.expand_dims(padded, axis=0))
        else:
            sir_gallahad_the_chaste.append(arr)
    return sir_gallahad_the_chaste

embeddings = np.concatenate(fix_jagged_2d(embeddings))
var_data['embeddings'] = embeddings
print(embeddings.shape)

In [ ]:
# create dataset for training SAE
def pad_sequences(amino_acid_seq:str, pad_length:int):
    if len(amino_acid_seq)==pad_length:
        return amino_acid_seq
    else:
        return amino_acid_seq+''.join(['-' for i in range(pad_length-len(amino_acid_seq))])

residue_embeddings = np.vstack(embeddings)
residue_labels = ''.join([pad_sequences(i, 148) for i in var_data['sequence']])
print(residue_embeddings.shape)
print(len(residue_labels))

In [ ]:
embed_data = pd.DataFrame(residue_embeddings)
embed_data["residues"] = [i for i in residue_labels]
ids = []
for x in var_data['sequence']:
    for n in range(148):
        ids.append(x)
embed_data["id"] = ids
embed_data

In [ ]:
embed_data.describe()

In [ ]:
# just in case we want to reload / use this later
embed_data.to_csv("./data/heme_SAE_train.csv")

### Training the SAE

In [ ]:
# cleaning some memory
del embed_data, embeddings, var_data

In [ ]:
# key parameters, model name
SAE_name = 'ESM-2-1280d-SAE-ef4'
embed_length = 1280
ef = 4

print("=== Initializing Model ===")
# initialize the optimizer
optimizer = keras.optimizers.Adam(learning_rate=0.001, beta_1=0.9, beta_2=0.98,
                                     epsilon=1e-9)

# initialize the loss function
loss = keras.losses.MeanSquaredError()

# initialize the metrics
metrics = [
    keras.metrics.MeanSquaredError(),
    keras.metrics.Metric(name='placeholder') # placeholder for training, feature output requires a 2nd metric to appease keras
]

# compile the model
autoencoder = SparseAutoEncoder(encoding_size=embed_length, expansion_factor=ef)
autoencoder.compile(optimizer=optimizer, loss=loss, metrics=metrics)

tb_callback = keras.callbacks.TensorBoard(log_dir=f'./logs/{SAE_name}', histogram_freq=5)
early_stopping = keras.callbacks.EarlyStopping(
    monitor='loss',
    min_delta=0.001,
    patience=10, 
    restore_best_weights=True
    )

print("=== Training Model ===")
history = autoencoder.fit(
    x=residue_embeddings,
    y=residue_embeddings,
    batch_size=32, 
    epochs=100, 
    validation_split=0.1, 
    callbacks=[tb_callback, early_stopping]
    )

print("=== Saving Model ===")
path = f'./models/{SAE_name}.keras'

autoencoder.save(path)
autoencoder.save_weights(f'./models/{SAE_name}.weights.h5') # workaround via weight saving / loading

print(f"Model saved to: {path}")
print(autoencoder.summary())

In [ ]:
import matplotlib.pyplot as plt

print("===== MSE =====")
plt.plot(history.history['mean_squared_error'])
plt.plot(history.history['val_mean_squared_error'])
plt.show()

print("===== LOSS =====")
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.show()